# CS156 Assignment #1: Implementation

Code for the reminder classifier. The write-up lives in `write-up.ipynb`.

The Telegram export and the CSVs contain personal messages, so they live in `data/` and are gitignored. 

## 1. Data

Turns the Telegram Desktop "Export chat history" HTML into a CSV with one row per unique message I sent:

- `id`: SHA-256 hash of the message text
- `message`: the raw text
- `has_remind_keyword`: 1 if the text matches `\bremind\w*`, else 0
- `timestamp`: ISO 8601
- `label`: empty until the labeling step (reminder = 1, other = 0)

Messages from the first days (assistant setup) and messages that are only a filler reply ("yes", "lgtm") are dropped.

In [2]:
import hashlib
import re
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

HTML_PATH = Path("data/messages.html")
CSV_PATH = Path("data/messages.csv")
USER_NAME = "Stiven Triana"
REMIND_PATTERN = re.compile(r"\bremind\w*", re.IGNORECASE)
TIMESTAMP_FORMAT = "%d %B %Y, %H:%M:%S"
CSV_COLUMNS = ["id", "message", "has_remind_keyword", "timestamp", "label"]
SETUP_DAYS = 2
FILLER_PHRASES = {
    "yes", "no", "y", "n", "yep", "yeah", "yup", "nope", "nah",
    "ok", "okay", "k", "kk", "sure", "go ahead", "do it", "sounds good",
    "lgtm", "thanks", "thank you", "thx", "ty", "cool", "nice", "great",
    "perfect", "done", "continue", "proceed", "correct", "right",
    "fine", "approved", "confirmed", "hey", "hi", "hello",
}

In [4]:
# 1. Read my messages out of the Telegram export
soup = BeautifulSoup(HTML_PATH.read_text(encoding="utf-8"), "html.parser")
messages = []
sender = ""
for message_div in soup.select("div.message.default"):
    name_div = message_div.select_one("div.from_name")
      # follow-up messages from the same sender leave the name out
    if name_div:
        sender = name_div.get_text().strip()
    text_div = message_div.select_one("div.body > div.text")
    # Skip messages from the assistant 
    if sender != USER_NAME or not text_div:
        continue

    # Telegram writes line breaks in a message as <br> tags, which get_text() would
    # drop and glue the lines together, so turn each one into a real newline first
    for line_break in text_div.find_all("br"):
        line_break.replace_with("\n")
    text = text_div.get_text().strip()
    # Skip messages with no text (e.g. photo's or audio messages), then save the
    # message with the time it was sent
    if text:
        sent_at = message_div.select_one("div.date")["title"]
        messages.append({"message": text, "timestamp": datetime.strptime(sent_at, TIMESTAMP_FORMAT)})

df = pd.DataFrame(messages)
print(f"user messages: {len(df)}")

# 2. Drop the first days (mostly setting up the assistant)
df = df[df["timestamp"] >= df["timestamp"].min() + timedelta(days=SETUP_DAYS)]
print(f"after dropping first {SETUP_DAYS} days: {len(df)}")

# 3. Drop messages that are only a filler reply like "yes" or "lgtm"
cleaned = df["message"].str.lower().str.replace(r"[^\w\s]", " ", regex=True).str.split().str.join(" ")
df = df[~cleaned.isin(FILLER_PHRASES)]
print(f"after dropping filler replies: {len(df)}")

# 4. Drop duplicates, and add the other columns
df = df.assign(id=df["message"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()))
df = df.drop_duplicates("id")
df["has_remind_keyword"] = df["message"].str.contains(REMIND_PATTERN).astype(int)
df["timestamp"] = df["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S")
df["label"] = ""
print(f"after dedup: {len(df)}")
print(f"has_remind_keyword=1: {df['has_remind_keyword'].sum()}")

# 5. Save, unless the CSV is already written and has labels
has_labels = CSV_PATH.exists() and (pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)["label"] != "").any()
if has_labels:
    print("csv already written and already has labels. Skipping overwrite")
else:
    df[CSV_COLUMNS].to_csv(CSV_PATH, index=False)
    print(f"\ncsv written to {CSV_PATH} with {len(df)} rows")

user messages: 679
after dropping first 2 days: 563
after dropping filler replies: 559
after dedup: 558
has_remind_keyword=1: 238

csv written to data\messages.csv with 558 rows


## 2. Labelling

Labels come from Perplexity's `pplx-decider` (27B decision model), served locally by Ollama, so the messages never leave this machine. A decision model doesn't generate text. It gets the message as `state` plus one typed `choice` question with two options, `reminder` and `other`, and returns a probability for each option. The more probable option becomes the label (`reminder` = 1, `other` = 0).

The "prompt" is the question in `LABEL_QUESTION`: its instructions and the description of each option. The examples in it are invented, not taken from my messages. If a request fails or the reply can't be parsed, it is retried up to `MAX_ATTEMPTS` times. Messages that still fail keep an empty label so a re-run picks them up. The run starts by reading the CSV and reporting how many rows are already labelled, then skips them. The CSV is saved after every message (atomically via a temp file, so an interrupt can't corrupt it), so the run can be stopped at any point and resumed by running the cell again. The Data cell also refuses to overwrite a CSV that already has labels.

Assumes Ollama (v0.35.0 or later) is running locally with the model pulled (`ollama pull pplx-decider`).

In [17]:
import time

import requests

# Start ollama with ollama serve
# If it gives an error:
# Error: listen tcp 127.0.0.1:11434: bind: Only one usage of each socket address (protocol/network address/port) is normally permitted.
# Then curl the address to verify it already is running there
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "pplx-decider"
MAX_ATTEMPTS = 3
REQUEST_TIMEOUT_SECONDS = 300
QUESTION_NAME = "reminder_request"
CHOICE_TO_LABEL = {"reminder": 1, "other": 0}
INSTRUCTIONS = "The state is a message a user sent to their personal AI assistant. Is it a request related to reminders? I.e., to set a reminder, change a reminder, or otherwise reminder related?"

REMINDER = "The user asks to be reminded of something at a specific time or after a delay, once or on a recurring schedule, or asks to create, change, list or cancel such reminders. Examples: 'remind me to clean my room at 21' or 'At 9, let me know to check whether my package has arrived' or 'what reminders have I set today?' or 'in 20min let me know the toaster is done'."

OTHER = "The user's request is NOT related to reminders, or it mentions words related to them but isn't a reminder ask. Examples: 'that reminds me of a song. who sings it?' or 'can you find if Best Buy closes before 21?' or 'i like this quote, save it: “Every sunset reminds me that endings can be beautiful.”'."

LABEL_QUESTION = {
    "type": "choice",
    "instructions": INSTRUCTIONS,
    "criteria": {"reminder": REMINDER, "other": OTHER},
}

In [18]:
def label_message(message: str) -> int | None:
    """Ask the model about one message. Returns 1, 0, or None if every attempt failed."""
    payload = {
        "model": OLLAMA_MODEL,
        "state": message,
        "questions": {QUESTION_NAME: LABEL_QUESTION},
    }
    # Retries just in case 
    for attempt in range(1, MAX_ATTEMPTS + 1):
        response_text = ""
        try:
            response = requests.post(
                f"{OLLAMA_URL}/v1/systemone", json=payload, timeout=REQUEST_TIMEOUT_SECONDS
            )
            response_text = response.text
            choice = response.json()["answers"][QUESTION_NAME]["choice"]
        except (requests.RequestException, ValueError, KeyError, TypeError) as error:
            print(f"\nattempt {attempt}/{MAX_ATTEMPTS} failed: {type(error).__name__}: {error} | response: {response_text[:200]!r}")
            continue

        label = CHOICE_TO_LABEL.get(choice)

        # Check the label is something
        if label is not None:
            return label
        print(f"\nattempt {attempt}/{MAX_ATTEMPTS}: unexpected choice {choice!r}")
    return None


def label_all_messages(csv_path: Path) -> int:
    """Label every row that has no label yet, saving after each one. Returns the failure count."""
    # Read the dataset
    dataset = pd.read_csv(csv_path, dtype=str, keep_default_na=False)

    # Check which rows have already been labelled
    already_labelled = dataset["label"].value_counts()
    print(
        f"already labelled: {already_labelled.get('1', 0)} reminder, "
        f"{already_labelled.get('0', 0)} other (skipped)"
    )

    total_rows = len(dataset)
    labelled_before = int((dataset["label"] != "").sum())
    labelled_now = 0
    failures = 0
    start_time = time.perf_counter()
    # Start labelling
    for row_index, row in dataset.iterrows():
        if row["label"] != "":
            continue

        # Label an individual message using a helper functino
        label = label_message(row["message"])

        # Count failures
        if label is None:
            failures += 1
            continue

        dataset.at[row_index, "label"] = str(label)
        dataset.to_csv(csv_path, index=False)

        # Count successes
        labelled_now += 1
        labelled_total = labelled_before + labelled_now
        print(f"\r{labelled_total}/{total_rows} labelled so far", end="", flush=True)
    print()
    print(f"labelling took {time.perf_counter() - start_time:.1f}s")
    return failures

In [19]:
failed = label_all_messages(CSV_PATH)
labels = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)["label"]
print(f"model: {OLLAMA_MODEL}")
print(f"label=1: {(labels == '1').sum()}, label=0: {(labels == '0').sum()}")
print(f"unlabelled after {MAX_ATTEMPTS} attempts each: {failed}")

already labelled: 0 reminder, 0 other (skipped)
558/558 labelled so far
labelling took 1929.5s
model: pplx-decider
label=1: 257, label=0: 301
unlabelled after 3 attempts each: 0


In [14]:
# Messages where the keyword regex and the model disagree
MAX_MESSAGE_CHARS = 200

labelled = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
disagreements = labelled[(labelled["has_remind_keyword"] == "1") != (labelled["label"] == "1")]

total = len(labelled)
regex_reminders = (labelled["has_remind_keyword"] == "1").sum()
model_reminders = (labelled["label"] == "1").sum()
print(f"regex: {regex_reminders} reminder, {total - regex_reminders} other")
print(f"model: {model_reminders} reminder, {total - model_reminders} other")
print(f"disagreement: {len(disagreements) / total:.1%}")
print(f"disagreements: {len(disagreements)} of {total}")
print()

# Opinions come first so they never scroll out of view
for number, (_, row) in enumerate(disagreements.iterrows(), start=1):
    regex_opinion = "reminder" if row["has_remind_keyword"] == "1" else "other"
    model_opinion = "reminder" if row["label"] == "1" else "other"
    text = " ".join(row["message"].split())[:MAX_MESSAGE_CHARS]
    print(f"{number:>2} | regex: {regex_opinion:<8} | model: {model_opinion:<8} | {text}")

regex: 238 reminder, 320 other
model: 257 reminder, 301 other
disagreement: 4.8%
disagreements: 27 of 558

 1 | regex: other    | model: reminder | Also add to my calendar this sunday to visit park avenue armory between 12 - 7pm
 2 | regex: other    | model: reminder | On my calendar, on august 2nd, add to plan a Patagonia trip with <redacted>
 3 | regex: other    | model: reminder | now done, stop that cron
 4 | regex: other    | model: reminder | and add one at 12:45 to go w/<redacted> team to eat here: https://www.jazzcultural.com
 5 | regex: other    | model: reminder | Add to my calendar pls Item Details Venue Subtotal Total <redacted> 1 x Ben Davis: Can San Francisco Become The Florence Of A New Renaissance? - Thursday, September 10th, 2026 at 6:00 PM PDT - Registr
 6 | regex: other    | model: reminder | Ok so, now onto the sauce: I want you to set up a daily cron at 9am that has you or a subagent check for deals on restaurants ! Basically, i want to get my moneys worth and chec

## 3. Split

80% of the messages go to training and 20% to testing. The split is stratified on `label`, so both groups keep the same share of reminders, and the seed is fixed so it is the same every run. The test set is only used for the final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2
RANDOM_SEED = 94102

dataset = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
train, test = train_test_split(
    dataset, test_size=TEST_SIZE, stratify=dataset["label"], random_state=RANDOM_SEED
)

for name, group in [("train", train), ("test", test)]:
    reminders = (group["label"] == "1").sum()
    print(f"{name}: {len(group)} messages ({reminders} reminder, {len(group) - reminders} other)")

train: 446 messages (205 reminder, 241 other)
test: 112 messages (52 reminder, 60 other)


# 4. Model Training 